### Vector stores and retrievers
This video tutorial will tamillarıze you with Langchain's vector store and retriever abstractions. I nese abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.
We will cover
• Documents
• Vector stores
• Retrievers

### Documents

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="This is a test document.",
        metadata={"source": "test_source"},
    ),
    Document(
        page_content="This is another test document.",
        metadata={"source": "test_source_2"},
    ),
]


In [3]:
documents


[Document(metadata={'source': 'test_source'}, page_content='This is a test document.'),
 Document(metadata={'source': 'test_source_2'}, page_content='This is another test document.')]

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.environ.get("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm= ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.7, max_output_tokens=512)

/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/pydantic/main.py:263: UserWarning: WARNING! max_output_tokens is not default parameter.
                    max_output_tokens was transferred to model_kwargs.
                    Please confirm that max_output_tokens is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6637.47it/s]


### Vector Store

In [6]:
## Vector Store
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore


In [7]:
vectorstore.similarity_search("test document", k=2)

[Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.'),
 Document(id='f22b7935-35a0-4e4e-802b-76dd8646e2d9', metadata={'source': 'test_source_2'}, page_content='This is another test document.')]

In [8]:
## async query

await vectorstore.asimilarity_search("test document", k=2)

[Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.'),
 Document(id='f22b7935-35a0-4e4e-802b-76dd8646e2d9', metadata={'source': 'test_source_2'}, page_content='This is another test document.')]

In [9]:
vectorstore.similarity_search_with_score("test document", k=2)

[(Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.'),
  0.4799341559410095),
 (Document(id='f22b7935-35a0-4e4e-802b-76dd8646e2d9', metadata={'source': 'test_source_2'}, page_content='This is another test document.'),
  0.5311816930770874)]

### Retriever

In [10]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["test", "code"])

[[Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.')],
 [Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.')]]

In [11]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)
retriever.batch(["test", "code"])


[[Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.')],
 [Document(id='ff1d813f-490a-45a3-9870-dc108f572c92', metadata={'source': 'test_source'}, page_content='This is a test document.')]]